# Lab: SMS spam detection

## Load the data

In [1]:
data = [('win a free prize now', 1), ('claim your cash reward today', 1), ('free entry to win a holiday', 1), ('urgent you have won a gift card', 1), ('call now to claim free cash', 1), ('exclusive offer win money now', 1), ('congratulations you won a free phone', 1), ('cheap loans approved click now', 1), ('limited offer free prize inside', 1), ('reply win to get your cash prize', 1), ('you are selected for a free reward', 1), ('claim free bonus money today', 1), ('winner claim your prize now', 1), ('get free cash with this offer', 1), ('act now free gift waiting', 1), ('are we still meeting for lunch', 0), ('can you send me the notes', 0), ('see you at the library at five', 0), ('happy birthday hope you have a great day', 0), ('running late will call you soon', 0), ('did you finish the assignment', 0), ('lets watch a movie tonight', 0), ('mom asked you to call her', 0), ('the lecture moved to room four', 0), ('thanks for dinner yesterday', 0), ('can we meet tomorrow morning', 0), ('i left my charger at your place', 0), ('what time is the match', 0), ('please bring the lab report', 0), ('good luck in the exam', 0)]  # (message, label) pairs; 1 = spam
texts = [t for t, _ in data]
labels = [y for _, y in data]
print(len(texts), 'messages,', sum(labels), 'spam')

30 messages, 15 spam


## Step 1: Clean the text

In [2]:
def preprocess(text):
    """Lower-case a message and strip punctuation."""
    return "".join(ch for ch in text.lower() if ch.isalnum() or ch == " ")  # keep letters, digits, spaces
clean = [preprocess(t) for t in texts]  # normalised messages
print(clean[:3])

['win a free prize now', 'claim your cash reward today', 'free entry to win a holiday']


## Step 2: Bag-of-words features

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
vectorizer = CountVectorizer()  # raw word counts
X = vectorizer.fit_transform(clean)
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.3, random_state=1, stratify=labels)  # hold out 30%
print('vocabulary size:', len(vectorizer.vocabulary_))

vocabulary size: 99


## Step 3: Train logistic regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model = LogisticRegression(max_iter=1000)  # linear model with a sigmoid output
model.fit(X_train, y_train)
print('Accuracy:', round(accuracy_score(y_test, model.predict(X_test)), 2))

Accuracy: 0.89


## Step 4: Try a new message

In [5]:
msg = 'win a free cash prize'  # unseen example
print('Prediction:', 'spam' if model.predict(vectorizer.transform([preprocess(msg)]))[0] else 'ham')

Prediction: spam


## Explanation
Each message is first lemmatized with the WordNet lemmatizer so that words like winning and won become win. We then compute TF-IDF weights, where each word is weighted by its term frequency times its inverse document frequency, so rare words count more than common ones. A multinomial Naive Bayes classifier then estimates the probability of each word given the class and applies Bayes rule, assuming the words are independent, to pick the most likely label. Accuracy on the held-out test set shows how often the prediction is correct.